In [ ]:
"""
# ==============================================================================
# EXP_45: Final Offline Submission Kernel for Kaggle "Name That LLM"
# Self-contained, leak-free production inference pipeline.
# Architecture: EXP_44-C Hybrid Prompt-Residual GAT + EXP_36 Soft Convex Fusion
# Post-Processing: Injective Bipartite Matching (Hungarian Assignment)
# Output Schema: ['id', 'model']
# ==============================================================================
"""

import os
import sys
import gc
import re
import time
import math
import random
import itertools
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import ledoit_wolf_shrinkage
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModel

# Pinned Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABELS = [
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra',
    'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'
]
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}
IDX_TO_LABEL = {i: l for i, l in enumerate(LABELS)}

STOPWORDS_70 = [
    'the','and','but','that','which','with','into','from','for','you','your','our','we','i','it','its','they','their',
    'there','this','these','those','however','therefore','moreover','although','because','perhaps','might','would',
    'could','should','like','such','just','very','also','not','if','or','as','so','by','of','to','in','on','at','is',
    'are','be','was','were','have','has','had','may','must','all','any','each','every','many','some','more','most',
    'only','even','still','then','when','where','why','how','what','who'
]

MARKERS_55 = [
    '--', '---', '—', '–', '-', ':', ';', '!', '?', '(', ')', '[', ']', '{', '}',
    '"', "'", '`', '``', "''", '“', '”', '‘', '’', '...', '…', '/', '\\', '@', '#',
    '$', '%', '^', '&', '*', '_', '=', '+', '<', '>', '~', '|',
    '**', '##', '###', '####', '- ', '* ', '1. ', '2. ', '> ', '```'
]

WRE = re.compile(r"[A-Za-z]+(?:[’'][A-Za-z]+)?")
SRE = re.compile(r'(?<=[.?!])\s+')


def resolve_model_path(model_identifier):
    """Resolves model path across Kaggle offline datasets and local HuggingFace cache."""
    short_name = model_identifier.split('/')[-1].lower()

    # 1. Search inside /kaggle/input if mounted on Kaggle
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files:
                root_lower = root.lower()
                matched = False
                
                # Check for MiniLM (Strictly exclude DeBERTa)
                if "minilm" in short_name:
                    if "deberta" in root_lower or "v3-small" in root_lower:
                        matched = False
                    elif "minilm" in root_lower or "all-mini" in root_lower:
                        matched = True
                    else:
                        try:
                            import json
                            with open(os.path.join(root, "config.json"), "r") as f:
                                cfg = json.load(f)
                            if cfg.get("hidden_size") == 384 and "deberta" not in str(cfg).lower():
                                matched = True
                        except Exception:
                            pass
                
                # Check for DeBERTa
                elif "deberta" in short_name:
                    if "deberta" in root_lower or "v3-small" in root_lower:
                        matched = True
                    else:
                        try:
                            import json
                            with open(os.path.join(root, "config.json"), "r") as f:
                                cfg = json.load(f)
                            if cfg.get("hidden_size") == 768 and ("deberta" in str(cfg).lower() or cfg.get("model_type") in ["deberta", "deberta-v2"]):
                                matched = True
                        except Exception:
                            pass

                if matched:
                    print(f">> [Offline Model Found] {model_identifier} -> {root}")
                    return root

    # 2. Check direct local directories
    candidates = [
        f"/kaggle/input/{model_identifier.split('/')[-1]}",
        f"/kaggle/input/{short_name}",
        f"/kaggle/input/{model_identifier.replace('/', '-')}"
    ]
    for c in candidates:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "config.json")):
            return c

    # 3. If running inside Kaggle environment and no local folder exists, return None (avoids DNS timeout)
    if os.path.exists("/kaggle"):
        return None

    return model_identifier


def find_pretrained_bundle(base_dir=None):
    """Searches /kaggle/input and local repository for pre-fitted EXP_45 pipeline bundle."""
    # 1. Search /kaggle/input
    if os.path.exists("/kaggle/input"):
        bundle_path = None
        gat_path = None
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                f_lower = f.lower()
                if f_lower in ["pipeline_bundle.joblib", "exp45_pipeline.joblib", "pipeline.joblib"]:
                    bundle_path = os.path.join(root, f)
                if f_lower in ["hybrid_gat.pt", "exp45_gat.pt", "gat.pt"]:
                    gat_path = os.path.join(root, f)
        if bundle_path:
            return bundle_path, gat_path

    # 2. Local search candidates
    candidate_dirs = [
        base_dir,
        os.getcwd(),
        os.path.dirname(os.path.abspath(__file__ if '__file__' in globals() else '.')),
        os.path.dirname(os.path.dirname(os.path.abspath(__file__ if '__file__' in globals() else '.'))),
        "/home/rj2826/name-the-llm"
    ]
    for d in candidate_dirs:
        if d:
            b = os.path.join(d, "models/exp45/pipeline_bundle.joblib")
            g = os.path.join(d, "models/exp45/hybrid_gat.pt")
            if os.path.exists(b):
                return b, (g if os.path.exists(g) else None)

    return None, None


def find_open_pool_path(base_dir=None):
    """Searches /kaggle/input and local repository for balanced Open Pool parquet data."""
    # 1. Search /kaggle/input recursively
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                f_lower = f.lower()
                if f_lower.endswith(".parquet") and ("open_pool" in f_lower or "pool" in f_lower):
                    candidate = os.path.join(root, f)
                    return candidate

    # 2. Local candidates
    candidate_dirs = [
        base_dir,
        os.getcwd(),
        os.path.dirname(os.path.abspath(__file__ if '__file__' in globals() else '.')),
        os.path.dirname(os.path.dirname(os.path.abspath(__file__ if '__file__' in globals() else '.'))),
        "/home/rj2826/name-the-llm"
    ]
    for d in candidate_dirs:
        if d:
            for fname in ["open_pool_36k.parquet", "open_pool_clean.parquet", "open_pool.parquet"]:
                for sub in ["dist", "data/external", "data"]:
                    p = os.path.join(d, sub, fname)
                    if os.path.exists(p):
                        return p
    return None


def sanitize_text(text):
    """Robust sanitization handling None, NaN, extreme lengths, and non-strings."""
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    if len(text) > 30000:
        text = text[:30000]
    return text


def to_casing_shape(text):
    text = re.sub(r'[A-Z]+', 'X', text)
    text = re.sub(r'[a-z]+', 'x', text)
    return text


def to_function_words(text):
    tokens = re.findall(r"\w+|[^\w\s]", text.lower())
    mapped = [t if t in STOPWORDS_70 or not t.isalnum() else 'CONTENT' for t in tokens]
    return " ".join(mapped)


def extract_stylometry_row(t):
    """Extracts 144d rule-based stylometric feature vector for a text passage."""
    t = sanitize_text(t)
    words = WRE.findall(t)
    lo = [w.lower() for w in words]
    n_words = max(len(words), 1)
    n_chars = max(len(t), 1)
    lines = t.splitlines()
    n_lines = max(len(lines), 1)
    
    word_lens = np.array([len(w) for w in words] or [0])
    sentences = [z for z in SRE.split(t) if z.strip()]
    sent_lens = np.array([len(WRE.findall(z)) for z in sentences] or [0])
    
    base = [
        np.log1p(n_words),
        np.log1p(n_chars),
        n_chars / n_words,
        n_lines / n_words,
        len(set(lo)) / n_words,
        float(np.mean(word_lens)),
        float(np.std(word_lens)),
        float(np.mean(sent_lens)),
        float(np.std(sent_lens)),
        float(np.max(sent_lens)),
        float(np.mean(word_lens >= 8)),
        sum(c.isupper() for c in t) / n_chars,
        sum(c.isdigit() for c in t) / n_chars,
        sum(ord(c) > 127 for c in t) / n_chars,
        sum(c.isspace() for c in t) / n_chars,
        sum(c == '\n' for c in t) / n_chars,
        sum(c in MARKERS_55 for c in t) / n_chars,
        len(sentences) / n_words
    ]
    base += [min(t.count(m) / n_words, 3.0) for m in MARKERS_55]
    cnt = {}
    for w in lo:
        cnt[w] = cnt.get(w, 0) + 1
    base += [cnt.get(w, 0) / n_words for w in STOPWORDS_70]
    
    arr = np.array(base, dtype=np.float32)
    if len(arr) < 144:
        arr = np.pad(arr, (0, 144 - len(arr)))
    elif len(arr) > 144:
        arr = arr[:144]
    return arr


def extract_minilm_l3(texts, batch_size=64, max_len=256, device=None):
    """Extracts 384d Layer 3 mean-pooled syntax embeddings using all-MiniLM-L6-v2."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model_path = resolve_model_path("sentence-transformers/all-MiniLM-L6-v2")
    if model_path is None:
        raise FileNotFoundError("MiniLM offline weights not attached under /kaggle/input.")
    is_local = os.path.isdir(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=is_local)
    encoder = AutoModel.from_pretrained(model_path, local_files_only=is_local).to(device)
    encoder.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = [sanitize_text(t) for t in texts[i:i + batch_size]]
            tok = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
            out = encoder(**tok, output_hidden_states=True)
            l3 = out.hidden_states[3]
            mask = tok["attention_mask"].unsqueeze(-1)
            mean_pool = (l3 * mask).sum(1) / mask.sum(1).clamp(min=1e-8)
            embeddings.append(mean_pool.cpu().numpy().astype(np.float32))
    res = np.vstack(embeddings)
    if res.shape[1] != 384:
        raise ValueError(f"Expected 384 dimensions from MiniLM-L3, but got {res.shape[1]}. Model path was: {model_path}")
    return res


def extract_deberta_v3(texts, batch_size=32, max_len=512, device=None):
    """Extracts 768d mean-pooled semantic embeddings using microsoft/deberta-v3-small."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model_path = resolve_model_path("microsoft/deberta-v3-small")
    if model_path is None:
        raise FileNotFoundError("DeBERTa offline weights not attached under /kaggle/input.")
    is_local = os.path.isdir(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=is_local)
    encoder = AutoModel.from_pretrained(model_path, local_files_only=is_local).to(device)
    encoder.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = [sanitize_text(t) for t in texts[i:i + batch_size]]
            tok = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
            mask = tok["attention_mask"].unsqueeze(-1)
            out = encoder(**tok)
            last_hidden = out.last_hidden_state
            mean_pool = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            embeddings.append(mean_pool.cpu().numpy().astype(np.float32))
    res = np.vstack(embeddings)
    if res.shape[1] != 768:
        raise ValueError(f"Expected 768 dimensions from DeBERTa-v3, but got {res.shape[1]}. Model path was: {model_path}")
    return res


def recover_prompt_clusters(val_texts, max_cluster_size=18, threshold=0.15):
    """Unsupervised blind prompt recovery via average linkage on content-word TF-IDF."""
    clean_texts = [sanitize_text(t) for t in val_texts]
    if len(clean_texts) <= 1:
        return np.zeros(len(clean_texts), dtype=int)
        
    vec = TfidfVectorizer(stop_words='english', min_df=1, max_df=0.15, sublinear_tf=True)
    try:
        X_content = vec.fit_transform(clean_texts)
        if X_content.shape[1] == 0:
            return np.arange(len(clean_texts))
    except ValueError:
        return np.arange(len(clean_texts))
        
    sim = (X_content @ X_content.T).toarray()
    dist = np.clip(1.0 - sim, 0.0, 1.0)
    np.fill_diagonal(dist, 0.0)
    
    condensed_dist = squareform(dist, checks=False)
    Z = linkage(condensed_dist, method='average')
    cluster_labels = fcluster(Z, t=1.0 - threshold, criterion='distance')
    
    unique_c, counts = np.unique(cluster_labels, return_counts=True)
    final_labels = cluster_labels.copy()
    next_id = max(unique_c) + 1
    
    for c, cnt in zip(unique_c, counts):
        if cnt > max_cluster_size:
            members = np.where(cluster_labels == c)[0]
            sub_dist = squareform(dist[np.ix_(members, members)], checks=False)
            sub_Z = linkage(sub_dist, method='average')
            sub_labels = fcluster(sub_Z, t=max_cluster_size, criterion='maxclust')
            for sl in np.unique(sub_labels):
                sub_idx = members[sub_labels == sl]
                final_labels[sub_idx] = next_id
                next_id += 1
                
    return final_labels


def hungarian_assignment(probs, cluster_labels, min_cluster_size=2):
    """Enforces injective bipartite matching within recovered prompt clusters.
    
    Robustness guarantees:
    1. Initializes with direct argmax (never zeros), avoiding artificial class-0 defaults.
    2. Supports arbitrary cluster sizes; for clusters > 18, matches the top-18 highest-confidence
       samples and retains argmax predictions for the remainder.
    3. Handles singleton/small clusters cleanly without degradation.
    """
    preds = np.argmax(probs, axis=1)
    for c in np.unique(cluster_labels):
        idx = np.where(cluster_labels == c)[0]
        k = len(idx)
        if k < min_cluster_size or k <= 1:
            continue
        elif k <= 18:
            cost_matrix = -probs[idx]
            r, c_ind = linear_sum_assignment(cost_matrix)
            preds[idx[r]] = c_ind
        else:
            # Cluster has > 18 samples: match the 18 most confident samples
            cluster_probs = probs[idx]
            max_p = np.max(cluster_probs, axis=1)
            top18_sub = np.argsort(-max_p)[:18]
            sub_idx = idx[top18_sub]
            cost_matrix = -probs[sub_idx]
            r, c_ind = linear_sum_assignment(cost_matrix)
            preds[sub_idx[r]] = c_ind
    return preds


# ==============================================================================
# Inlined Shared-Prompt Gaussian Group Head
# ==============================================================================
GAUSS_DEFAULT = {
    "red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw",
    "lam_floor": 1e-4, "n_exact": 3, "inj": True, "iters": 5, "repel": True, "sp_scale": 1.0
}
_MASK = {}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def _log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]
    return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1]))
    cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y)
    mu = np.zeros((18, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]
        P = np.zeros_like(P)
        np.add.at(P, pinv, R)
        P /= np.maximum(cnt_p[:, None], 1.0)
    return mu, P, cnt_p

def gauss_fit(X, y, prompts, cfg=None):
    c = dict(GAUSS_DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    Pm = np.zeros((G, D))
    np.add.at(Pm, pinv, Xs)
    Pm /= np.maximum(np.bincount(pinv)[:, None], 1.0)
    R = Xs - Pm[pinv]
    d = int(min(c["d"], N - G - 1, D))
    if N < D:
        w, U = np.linalg.eigh(R @ R.T)
        o = np.argsort(w)[::-1][:d]
        w, U = np.clip(w[o], 1e-12, None), U[:, o]
        V = (R.T @ U) / np.sqrt(w)
    else:
        _, _, Vt = np.linalg.svd(R, full_matrices=False)
        V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"])
    Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0)
    Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / np.maximum(ng, 1.0)) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)
    Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se + 1e-6 * np.eye(d))
    Li = np.linalg.inv(L)
    M = Li @ Sp @ Li.T
    M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, c["lam_floor"], None)
    A = Q.T @ Li
    W = V @ A.T
    return {
        "mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0, "cfg": c
    }

def gauss_transform(X, prm):
    return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

def _singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]
    return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def _group_logpost(U, prm, cfg=None):
    n = len(U)
    m, lam = prm["m"], prm["lam"]
    if n == 1:
        return _log_softmax(_singleton_ll(U, m, lam))
    v = lam / (1 + (n - 1) * lam)
    L = _singleton_ll(U, m, lam)
    q = np.exp(_log_softmax(L))
    for _ in range(5):
        s = U - q @ m
        S = s.sum(0)
        Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]
        L = -0.5 * (R ** 2 / (1 + v)).sum(-1)
        lo = np.log(np.clip(1 - q, 1e-6, None))
        L = L + (lo.sum(0)[None] - lo)
        q = np.exp(_log_softmax(L))
    return _log_softmax(L)

def gauss_scores(X, lab, prm, U=None):
    U = gauss_transform(X, prm) if U is None else U
    n = len(U)
    out = np.empty((n, prm["m"].shape[0]))
    pu, pinv = np.unique(lab, return_inverse=True)
    sizes = np.bincount(pinv)
    for g in np.unique(pinv):
        ii = np.where(pinv == g)[0]
        out[ii] = _group_logpost(U[ii], prm)
    return out


# ==============================================================================
# Relational Graph Attention Network (GAT)
# ==============================================================================
def compute_prompt_residuals(X, prompt_labels):
    R = np.zeros_like(X)
    for p in np.unique(prompt_labels):
        mask = (prompt_labels == p)
        mean_p = np.mean(X[mask], axis=0, keepdims=True)
        R[mask] = X[mask] - mean_p
    return R


def pack_prompt_graph_batch(node_features, prompt_labels, node_targets=None, max_k=18):
    unique_prompts = np.unique(prompt_labels)
    n_prompts = len(unique_prompts)
    D = node_features.shape[1]

    batch_x = np.zeros((n_prompts, max_k, D), dtype=np.float32)
    batch_mask = np.zeros((n_prompts, 1, max_k, max_k), dtype=bool)
    batch_y = np.full((n_prompts, max_k), -1, dtype=np.int64)
    batch_valid = np.zeros((n_prompts, max_k), dtype=bool)
    index_mapping = []

    for p_idx, p in enumerate(unique_prompts):
        orig_indices = np.where(prompt_labels == p)[0]
        k = min(len(orig_indices), max_k)
        batch_x[p_idx, :k] = node_features[orig_indices[:k]]
        batch_mask[p_idx, 0, :k, :k] = True
        batch_valid[p_idx, :k] = True
        if node_targets is not None:
            batch_y[p_idx, :k] = node_targets[orig_indices[:k]]
        index_mapping.append((p, orig_indices[:k]))

    return (
        torch.tensor(batch_x, dtype=torch.float32),
        torch.tensor(batch_mask, dtype=torch.bool),
        torch.tensor(batch_y, dtype=torch.long),
        torch.tensor(batch_valid, dtype=torch.bool),
        index_mapping
    )


class PromptGATLayer(nn.Module):
    def __init__(self, d_in, d_out, n_heads=4, dropout=0.15):
        super().__init__()
        self.d_out = d_out
        self.n_heads = n_heads
        self.head_dim = d_out // n_heads
        assert d_out % n_heads == 0, "d_out must be divisible by n_heads"

        self.q_proj = nn.Linear(d_in, d_out, bias=False)
        self.k_proj = nn.Linear(d_in, d_out, bias=False)
        self.v_proj = nn.Linear(d_in, d_out, bias=False)
        self.out_proj = nn.Linear(d_out, d_out)
        self.skip_proj = nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity()

        self.norm1 = nn.LayerNorm(d_out)
        self.norm2 = nn.LayerNorm(d_out)
        self.ffn = nn.Sequential(
            nn.Linear(d_out, d_out * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_out * 2, d_out),
            nn.Dropout(dropout)
        )
        self.dropout = dropout

    def forward(self, x, mask=None):
        B, K, _ = x.shape
        q = self.q_proj(x).view(B, K, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, K, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, K, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, dropout_p=self.dropout if self.training else 0.0
        )
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, K, self.d_out)
        h = self.norm1(self.skip_proj(x) + self.out_proj(attn_out))
        out = self.norm2(h + self.ffn(h))
        return out


class PromptResidualGAT(nn.Module):
    def __init__(self, in_features=1296, hidden_dim=128, num_classes=18, n_heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.gat1 = PromptGATLayer(hidden_dim, hidden_dim, n_heads=n_heads, dropout=dropout)
        self.gat2 = PromptGATLayer(hidden_dim, hidden_dim, n_heads=n_heads, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, mask=None):
        h = self.input_proj(x)
        h = self.gat1(h, mask=mask)
        h = self.gat2(h, mask=mask)
        logits = self.classifier(h)
        return logits


class FinalSubmissionPipeline:
    """Production Hybrid Pipeline: EXP_36 (SVC + Gauss) + EXP_44-C Hybrid GAT + Hungarian."""

    def __init__(self, alpha=0.05, w_gauss=0.05, w_pool=0.35, device=None):
        self.alpha = alpha
        self.w_gauss = w_gauss
        self.w_pool = w_pool
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.vec_char = None
        self.vec_word = None
        self.vec_shape = None
        self.vec_func = None
        self.clf_svc = None
        self.gauss_prm = None
        self.gat_model = None
        self.scaler = None
        self.is_fitted = False

    @classmethod
    def from_pretrained(cls, bundle_path, gat_path=None, device=None):
        import joblib
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        bundle = joblib.load(bundle_path)
        
        pipeline = cls(
            alpha=bundle.get('alpha', 0.05),
            w_gauss=bundle.get('w_gauss', 0.05),
            w_pool=bundle.get('w_pool', 0.35),
            device=device
        )
        pipeline.vec_char = bundle['vec_char']
        pipeline.vec_word = bundle['vec_word']
        pipeline.vec_shape = bundle['vec_shape']
        pipeline.vec_func = bundle['vec_func']
        pipeline.clf_svc = bundle['clf_svc']
        pipeline.gauss_prm = bundle['gauss_prm']
        pipeline.scaler = bundle.get('scaler', None)
        pipeline.view_weights = bundle.get('view_weights', (1.0, 1.0, 1.0, 1.0))
        
        # Verify offline transformer availability for test inference
        minilm_path = resolve_model_path("sentence-transformers/all-MiniLM-L6-v2")
        deberta_path = resolve_model_path("microsoft/deberta-v3-small")
        pipeline.has_minilm = minilm_path is not None
        pipeline.has_deberta = deberta_path is not None
        pipeline.has_gat = bundle.get('has_gat', True) and (pipeline.has_minilm or pipeline.has_deberta)

        if pipeline.has_gat and gat_path and os.path.exists(gat_path):
            try:
                gat_model = PromptResidualGAT(in_features=1296, hidden_dim=128, num_classes=18, n_heads=4, dropout=0.2).to(device)
                gat_model.load_state_dict(torch.load(gat_path, map_location=device))
                gat_model.eval()
                pipeline.gat_model = gat_model
                print(">> [Model Loaded] Hybrid Prompt-Residual GAT (1296d) active.")
            except Exception as e:
                print(f">> [Warning] Failed to load GAT weights ({e}). Disabling GAT.")
                pipeline.has_gat = False
        else:
            pipeline.gat_model = None
            pipeline.has_gat = False

        pipeline.is_fitted = True
        return pipeline

    def fit(self, train_df, open_pool_df=None):
        print(f">> Fitting FinalSubmissionPipeline on {len(train_df)} official samples on {self.device}...")
        t0 = time.time()

        train_texts = [sanitize_text(t) for t in train_df["text"].values]
        y_train = np.array([LABEL_TO_IDX[m] for m in train_df["model"].values])
        prompts_train = train_df["prompt_id"].values

        all_texts = list(train_texts)
        all_y = list(y_train)
        sample_weights = [1.0] * len(train_texts)

        if open_pool_df is not None and len(open_pool_df) > 0:
            pool_texts = [sanitize_text(t) for t in open_pool_df["text"].values]
            pool_y = [LABEL_TO_IDX[m] for m in open_pool_df["model"].values]
            all_texts.extend(pool_texts)
            all_y.extend(pool_y)
            sample_weights.extend([self.w_pool] * len(pool_texts))

        all_y = np.array(all_y)
        sample_weights = np.array(sample_weights)

        # 4-View Vectorizers
        print(">> Fitting 4-view TF-IDF vectorizers...")
        self.vec_char = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3, max_features=120000, sublinear_tf=True)
        self.vec_word = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=2, max_features=80000, sublinear_tf=True, token_pattern=r'(?u)\b\w+\b|[^\w\s]')
        self.vec_shape = TfidfVectorizer(analyzer='char', ngram_range=(2, 6), min_df=3, max_features=40000, sublinear_tf=True)
        self.vec_func = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=2, max_features=30000, sublinear_tf=True)

        self.view_weights = (1.0, 0.7, 0.5, 0.5)
        w_c, w_w, w_s, w_f = self.view_weights
        X_char = self.vec_char.fit_transform(all_texts) * w_c
        X_word = self.vec_word.fit_transform(all_texts) * w_w
        X_shape = self.vec_shape.fit_transform([to_casing_shape(t) for t in all_texts]) * w_s
        X_func = self.vec_func.fit_transform([to_function_words(t) for t in all_texts]) * w_f
        X_svc = hstack([X_char, X_word, X_shape, X_func]).tocsr()

        print(f">> Fitting LinearSVC(C=0.5) on shape {X_svc.shape}...")
        self.clf_svc = LinearSVC(C=0.5, class_weight='balanced', loss='squared_hinge', max_iter=2000, random_state=42)
        self.clf_svc.fit(X_svc, all_y, sample_weight=sample_weights)

        print(">> Extracting syntax, semantic, and stylometric features for Gaussian Head & GAT...")
        self.has_minilm = False
        try:
            X_syn_train = extract_minilm_l3(train_texts, device=self.device)
            self.has_minilm = True
        except Exception as e:
            print(f">> [Notice] MiniLM offline weights not detected ({e}). Using fallback.")
            X_syn_train = np.zeros((len(train_texts), 384), dtype=np.float32)

        self.has_deberta = False
        try:
            X_deb_train = extract_deberta_v3(train_texts, device=self.device)
            self.has_deberta = True
        except Exception as e:
            print(f">> [Notice] DeBERTa offline weights not detected ({e}). Using fallback.")
            X_deb_train = np.zeros((len(train_texts), 768), dtype=np.float32)

        X_sty_train = np.stack([extract_stylometry_row(t) for t in train_texts])
        self.has_gat = self.has_minilm or self.has_deberta

        # Shared-Prompt Gaussian Head
        X_dense_train = np.hstack([X_syn_train, X_sty_train]) if self.has_minilm else X_sty_train
        print(f">> Fitting Shared-Prompt Gaussian Head on {X_dense_train.shape}...")
        self.gauss_prm = gauss_fit(X_dense_train, y_train, prompts_train, cfg={'d': 128})

        # Hybrid Prompt-Residual GAT (1296d)
        if self.has_gat:
            X_hybrid_train = np.hstack([X_syn_train, X_deb_train, X_sty_train]).astype(np.float32)
            R_train = compute_prompt_residuals(X_hybrid_train, prompts_train)
            self.scaler = StandardScaler()
            R_train_s = self.scaler.fit_transform(R_train)

            bx_tr, bmask_tr, by_tr, bvalid_tr, _ = pack_prompt_graph_batch(R_train_s, prompts_train, y_train)
            bx_tr, bmask_tr, by_tr, bvalid_tr = bx_tr.to(self.device), bmask_tr.to(self.device), by_tr.to(self.device), bvalid_tr.to(self.device)

            print(f">> Training Hybrid GAT (1296d -> 128d, 4 heads) for 40 epochs on {len(np.unique(prompts_train))} prompt graphs...")
            torch.manual_seed(SEED)
            self.gat_model = PromptResidualGAT(in_features=1296, hidden_dim=128, num_classes=18, n_heads=4, dropout=0.2).to(self.device)
            optimizer = AdamW(self.gat_model.parameters(), lr=1e-3, weight_decay=1e-4)
            scheduler = CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)

            self.gat_model.train()
            for epoch in range(40):
                optimizer.zero_grad()
                logits = self.gat_model(bx_tr, bmask_tr)
                loss = F.cross_entropy(logits[bvalid_tr], by_tr[bvalid_tr])
                loss.backward()
                optimizer.step()
                scheduler.step()

            self.gat_model.eval()
        else:
            print(">> [Notice] Operating on 4-View LinearSVC + 144d Stylometry Gaussian Head + Hungarian Assignment.")

        self.is_fitted = True
        print(f">> Pipeline successfully fitted in {time.time() - t0:.1f}s.")
        return self

    def predict_proba(self, test_df):
        if not self.is_fitted:
            raise RuntimeError("Pipeline must be fitted before calling predict.")

        test_texts = [sanitize_text(t) for t in test_df["text"].values]
        n_test = len(test_texts)
        if n_test == 0:
            return np.zeros((0, len(LABELS))), np.array([])

        # 1. LinearSVC scoring
        w_c, w_w, w_s, w_f = getattr(self, 'view_weights', (1.0, 1.0, 1.0, 1.0))
        X_char = self.vec_char.transform(test_texts) * w_c
        X_word = self.vec_word.transform(test_texts) * w_w
        X_shape = self.vec_shape.transform([to_casing_shape(t) for t in test_texts]) * w_s
        X_func = self.vec_func.transform([to_function_words(t) for t in test_texts]) * w_f
        X_svc = hstack([X_char, X_word, X_shape, X_func]).tocsr()

        dec = self.clf_svc.decision_function(X_svc)
        if dec.ndim == 1:
            dec = np.vstack([-dec, dec]).T
        log_p_svc = dec - np.log(np.sum(np.exp(dec - np.max(dec, axis=1, keepdims=True)), axis=1, keepdims=True)) - np.max(dec, axis=1, keepdims=True)

        # 2. Prompt recovery
        test_clusters = recover_prompt_clusters(test_texts)

        # 3. Dense features extraction
        if self.has_minilm:
            try:
                X_syn_test = extract_minilm_l3(test_texts, device=self.device)
            except Exception:
                X_syn_test = np.zeros((n_test, 384), dtype=np.float32)
        else:
            X_syn_test = np.zeros((n_test, 384), dtype=np.float32)

        if self.has_deberta:
            try:
                X_deb_test = extract_deberta_v3(test_texts, device=self.device)
            except Exception:
                X_deb_test = np.zeros((n_test, 768), dtype=np.float32)
        else:
            X_deb_test = np.zeros((n_test, 768), dtype=np.float32)

        X_sty_test = np.stack([extract_stylometry_row(t) for t in test_texts])

        # 4. Gaussian Head scoring
        expected_gauss_dim = self.gauss_prm["mx"].shape[0] if (self.gauss_prm is not None and "mx" in self.gauss_prm) else None
        has_valid_minilm = (X_syn_test is not None and X_syn_test.shape[1] == 384 and np.any(X_syn_test != 0))

        if has_valid_minilm and expected_gauss_dim == 528:
            X_dense_test = np.hstack([X_syn_test, X_sty_test])
            log_p_gauss = gauss_scores(X_dense_test, test_clusters, self.gauss_prm)
            log_blend_36 = log_p_svc + self.w_gauss * log_p_gauss
            p_exp36 = softmax(log_blend_36, axis=1)
        elif not has_valid_minilm and expected_gauss_dim == 144:
            X_dense_test = X_sty_test
            log_p_gauss = gauss_scores(X_dense_test, test_clusters, self.gauss_prm)
            log_blend_36 = log_p_svc + self.w_gauss * log_p_gauss
            p_exp36 = softmax(log_blend_36, axis=1)
        else:
            p_exp36 = softmax(log_p_svc, axis=1)

        # 5. Hybrid Prompt-Residual GAT scoring if active
        if self.has_gat and self.gat_model is not None:
            try:
                X_hybrid_test = np.hstack([X_syn_test, X_deb_test, X_sty_test]).astype(np.float32)
                R_test = compute_prompt_residuals(X_hybrid_test, test_clusters)
                R_test_s = self.scaler.transform(R_test)

                bx_te, bmask_te, _, bvalid_te, mapping_te = pack_prompt_graph_batch(R_test_s, test_clusters)
                bx_te, bmask_te = bx_te.to(self.device), bmask_te.to(self.device)

                with torch.no_grad():
                    logits_gat = self.gat_model(bx_te, bmask_te)
                    probs_gat_tensor = F.softmax(logits_gat, dim=-1)

                p_gat = np.zeros((n_test, 18), dtype=np.float32)
                for p_idx, (_, orig_idx) in enumerate(mapping_te):
                    k = len(orig_idx)
                    p_gat[orig_idx] = probs_gat_tensor[p_idx, :k].cpu().numpy()

                p_final = (1.0 - self.alpha) * p_exp36 + self.alpha * p_gat
            except Exception as e:
                print(f">> [Notice] GAT inference bypassed ({e}). Using base predictions.")
                p_final = p_exp36
        else:
            p_final = p_exp36

        return p_final, test_clusters

    def predict(self, test_df):
        if len(test_df) == 0:
            return pd.DataFrame({"id": [], "model": []})

        probs, clusters = self.predict_proba(test_df)
        pred_indices = hungarian_assignment(probs, clusters)
        pred_labels = [IDX_TO_LABEL[idx] for idx in pred_indices]

        sub = pd.DataFrame({
            "id": test_df["id"].values,
            "model": pred_labels
        })
        return sub


def main():
    print("=" * 80)
    print("EXP_45: KAGGLE OFFLINE SUBMISSION EXECUTION")
    print("=" * 80)

    base_dir = None
    data_dir = None
    work_dir = "/kaggle/working" if os.path.exists("/kaggle") else "submissions"

    # 1. Discover Data Paths (Kaggle mount vs local repo)
    for cand in [
        "/kaggle/input/competitions/name-that-llm",
        "/kaggle/input/name-that-llm",
        "/kaggle/input/name-the-llm",
    ]:
        if os.path.exists(os.path.join(cand, "test.csv")):
            data_dir = cand
            break

    if data_dir is None and os.path.exists("/kaggle/input"):
        # Search /kaggle/input for test.csv
        for root, dirs, files in os.walk("/kaggle/input"):
            if "test.csv" in files:
                data_dir = root
                break

    if data_dir is None:
        # Local environment (safe against Jupyter NameError)
        try:
            base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        except NameError:
            cwd = os.getcwd()
            if os.path.exists(os.path.join(cwd, "data", "test.csv")):
                base_dir = cwd
            elif os.path.exists(os.path.join(os.path.dirname(cwd), "data", "test.csv")):
                base_dir = os.path.dirname(cwd)
            else:
                base_dir = cwd
        data_dir = os.path.join(base_dir, "data")
        work_dir = os.path.join(base_dir, "submissions")

    os.makedirs(work_dir, exist_ok=True)
    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")

    # Dynamic Component Detection
    bundle_path, gat_path = find_pretrained_bundle(base_dir)
    pool_path = find_open_pool_path(base_dir)
    minilm_path = resolve_model_path("sentence-transformers/all-MiniLM-L6-v2")
    deberta_path = resolve_model_path("microsoft/deberta-v3-small")

    has_bundle = bundle_path is not None
    has_pool = pool_path is not None
    has_minilm = minilm_path is not None
    has_deberta = deberta_path is not None

    print("\n" + "=" * 80)
    print("EXP_45 PIPELINE DIAGNOSTIC REPORT")
    print("=" * 80)
    print(f"[{'✓' if has_bundle else '✗'}] Pretrained Model Bundle:  {bundle_path if has_bundle else 'NOT FOUND (Will fit on the fly)'}")
    print(f"[{'✓' if has_pool else '✗'}] Balanced Open Pool (36k): {pool_path if has_pool else 'NOT FOUND'}")
    print(f"[{'✓' if has_minilm else '✗'}] Offline MiniLM Model:     {minilm_path if has_minilm else 'NOT FOUND'}")
    print(f"[{'✓' if has_deberta else '✗'}] Offline DeBERTa Model:    {deberta_path if has_deberta else 'NOT FOUND'}")
    print("-" * 80)

    if (has_bundle or has_pool) and has_minilm and has_deberta:
        mode = "FULL HYBRID ATTRIBUTION (EXP_44-C / EXP_45)"
        expected_ba = "0.6216 (0-DoF) / 0.6264 (OOF-selected)"
    elif (has_bundle or has_pool):
        mode = "LIGHTWEIGHT ATTRIBUTION (4-View SVC + 144d Stylometry Gaussian)"
        expected_ba = "~0.536"
    else:
        mode = "FALLBACK OFFICIAL-ONLY ATTRIBUTION (LinearSVC only)"
        expected_ba = "~0.348 (Baseline)"

    print(f"ACTIVE EXECUTION REGIME:    {mode}")
    print(f"EXPECTED BALANCED ACCURACY: {expected_ba}")
    print("=" * 80 + "\n")

    if not os.path.exists(test_path):
        raise FileNotFoundError(f"test.csv not found in {data_dir}!")

    test_df = pd.read_csv(test_path)

    # 2. Pipeline Initialization & Fitting
    # If Balanced Open Pool is available, prefer native fitting in Kaggle (~80s)
    # This guarantees 100% environment compatibility and completely eliminates scikit-learn version mismatch.
    if has_pool:
        print(f">> [Native Training Mode] Balanced Open Pool detected at {pool_path}.")
        print(">> Fitting pipeline directly in Kaggle runtime (~80s) for exact scikit-learn environment compatibility...")
        if not os.path.exists(train_path):
            raise FileNotFoundError(f"train.csv not found in {data_dir}!")
        train_df = pd.read_csv(train_path)
        pool_df = pd.read_parquet(pool_path)
        pipeline = FinalSubmissionPipeline(alpha=0.05, w_gauss=0.05, w_pool=0.35)
        pipeline.fit(train_df, pool_df)
    elif has_bundle:
        print(f">> [Fast Inference Mode] Loading pretrained model bundle from {bundle_path}...")
        pipeline = FinalSubmissionPipeline.from_pretrained(bundle_path, gat_path)
    else:
        print(">> [Fallback Mode] Fitting official-only training data...")
        if not os.path.exists(train_path):
            raise FileNotFoundError(f"train.csv not found in {data_dir}!")
        train_df = pd.read_csv(train_path)
        pipeline = FinalSubmissionPipeline(alpha=0.05, w_gauss=0.05, w_pool=0.35)
        pipeline.fit(train_df, None)

    print(f"\n>> Scoring {len(test_df)} test samples...")
    t0 = time.time()
    sub_df = pipeline.predict(test_df)
    print(f">> Inference finished in {time.time() - t0:.2f}s.")

    # 3. Output files for Kaggle submission
    out_parquet = os.path.join(work_dir, "submission.parquet")
    out_csv = os.path.join(work_dir, "submission.csv")
    sub_df.to_parquet(out_parquet, index=False)
    sub_df.to_csv(out_csv, index=False)
    print(f">> Saved: {out_parquet}")
    print(f">> Saved: {out_csv}")

    # Also ensure present in current working directory
    if not os.path.exists("submission.parquet"):
        sub_df.to_parquet("submission.parquet", index=False)
    if not os.path.exists("submission.csv"):
        sub_df.to_csv("submission.csv", index=False)

    print("\nSubmission Preview:")
    print(sub_df.head(10))
    print("=" * 80)
    print("SUBMISSION READY FOR KAGGLE")
    print("=" * 80)


if __name__ == "__main__":
    main()
